In [ ]:
from src.__entro_plot__ import *
from src.__models__ import *
from common.__common__ import *
import scienceplots


%matplotlib inline
print(plt.style.available)
plt.style.use(['science', 'no-latex'])
current_dir = os.getcwd() 
save_dir = f"D:{kPSep}Maks{kPSep}EntropyMarcosLev{kPSep}Python{kPSep}"

# read the log file for given set of parameters
N = 20
su=True
# directory = current_dir + f"{kPSep}Data{kPSep}Data_Quater{kPSep}resultsXYZ{N}{kPSep}"
directory = save_dir + "Data" + kPSep + "Sweep" + kPSep + f"resultsXYZ{N}{kPSep}"
# directory 
directory_s = save_dir

# READ THE ENTROPIES AND FIT THE AVERAGE

In [ ]:
directory_scaling = lambda L: save_dir + "Data" + kPSep + f"resultsXYZ{L:.1f}{kPSep}"
param_a = 'da'
param_b = 'Jb'

def dos_fit_L(L):
    def dos_fit(x, E_0, sig):
        return 1/np.sqrt(np.pi) / sig * np.sqrt(L) * np.exp(-L*(x-E_0)**2 / (sig**2))
    return dos_fit

def dos_fit_no_L(x, E_0, sig):
    return 1/np.sqrt(np.pi) / sig * np.exp(-(x-E_0)**2 / (sig**2))

def entro_fit_p(L, sig, E_0):
    def entro_fit(x, S_m, c):
        return np.array(S_m -  L * c * ((np.array(x)-E_0)**2) / sig**2) 
    return entro_fit

def analytic_S(c, sig, E0, L, Emin, Emax):
    constant = -0.5 * c - c / sig / sig * E0 * E0 * L + 2 * c / sig / sig * E0 * E0 * L
    
    x = (E0 - Emin)/sig if Emin is not None else 0
    y = (Emax - E0)/sig if Emax is not None else 0
    
    corr = c / 2 / np.sqrt(np.pi) / sig / (erf(x*np.sqrt(L)) + erf(y*np.sqrt(L))) if Emin is not None else 0
    A = -E0 * corr * 2. * np.exp(-(x**2) * np.sqrt(L)) if Emin is not None else 0
    B = E0 * corr * 2. * np.exp(-(y**2) * np.sqrt(L)) if Emax is not None else 0
    
    C = -2.0 * corr * np.sqrt(L) * (E0 + Emin) * np.exp(-(x**2)* L) if Emin is not None else 0
    D = 2.0 * corr * np.sqrt(L) * (E0 + Emax) * np.exp(-(y**2)* L) if Emax is not None else 0
    return constant + (A + B + C + D)

frac = 0.5

Ls = [20, 22, 24]
bino = np.int64(binom(Ls[-1], Ls[-1]//2))
print("N=", bino)
energy_concat = np.zeros(bino)
entro_concat = np.zeros(bino)
to_fig = { L:[] for L in Ls}
points = { L:[] for L in Ls}

Nh = 0
for L in Ls:
    folder_save = save_dir + kPSep + "plot_data" + kPSep + 'analytic' + kPSep + f'L={L}' + kPSep
    createFolder([folder_save])
    
    # read the DF with all files
    df_sweep = get_log_file(directory_scaling(L), read_log=False,su=su)
    df_sweep = df_sweep[df_sweep[param_b] == 2.0][df_sweep[param_a] == 0.2].reset_index().drop('index', axis = 1)

    # takes the Hilbert space sizes
    hilbert_sizes = df_sweep.loc[:,'Nh'].to_list()
    # takes the names of the models
    model_shorts = df_sweep.loc[:,'model_short'].to_list()
    
    columns = ['var(E)', 'sig', 'E0', 'c', 'Smax', 'S_max_true', 'S_av_analytical', 'S_av_numerical', f'S_v', f'S_v_a']
    df_save = np.zeros((len(model_shorts), len(columns)))
    df_save = pd.DataFrame(df_save, index=model_shorts, columns=columns)
    
    # enumerate sectors
    end = len(model_shorts)
    for i, short in enumerate(model_shorts):
        if i == end: break
        fig, ax = plt.subplots(3, figsize = (8, 16), dpi = 200)
        ax[1].set_xlabel("$E/L$", fontsize = 16)
        ax[1].set_ylabel("$S(E)$", fontsize = 16)
        ax[0].set_ylabel(r"$\rho (E)$", fontsize = 16)
        N = hilbert_sizes[i]
        
        # read the energy
        filename = short + '.h5'
        if not 'spectrum' in short:
            filename = short + ',spectrum_num=' + str(int(hilbert_sizes[i])) + '.h5'
        
        # read all entropies
        ent, av_idx, Nh, _, dictionary = read_entropies(directory_scaling(L), filename, 1.0)
        ent = np.array(ent.iloc[-1])
        
        # get energy density
        en = np.array(read_h5_file(directory_scaling(L), filename, 'energy')).flatten() / L
        
        # ---------------------------- calculate the DOS ----------------------------
        dos, bins = np.histogram(en, bins=100, normed=True)
        parameters, covariance = curve_fit(dos_fit_L(L), np.array(bins[1:]), np.array(dos, dtype=np.float32))
        # parameters, covariance = curve_fit(dos_fit_no_L, np.array(bins[1:]), np.array(dos, dtype=np.float32))
        
        # beta = float(parameters[1])
        sig = np.abs(parameters[1])
        sig_other = np.std(en*np.sqrt(L))
        E0 = float(parameters[0])
        
        ax[0].stairs(dos, bins, label = 'LDOS', lw = 2)
        ax[0].plot(en, dos_fit_L(L)(en, E0, sig), label = f'$N \exp (-L(x-({E0:.3f}))^2/({sig:.3f}))^2$', lw = 2)
        # ax[0].plot(en * L, dos_fit_no_L(en * L, E0, sig), label = f'$\exp (-L(x-({E0:.3f}))^2/({sig:.3f}))^2$', lw = 2)

        # ---------------------------- calculate the entropies ----------------------------
        S_m =  np.max(ent)
        #e_fit = lambda x, S_m, c: np.array(S_m - L * c * (np.power((x-E0), 2) / sig**2)) 
        e_fit = lambda x, c: np.array(S_m - L * c * (np.power((x-E0), 2) / sig**2)) 

        parameters, covariance = curve_fit(e_fit, np.array(en), np.array(ent))
        
        # S_max = parameters[0]
        S_max = S_m

        c = parameters[0]#parameters[1]
        Emax = np.sqrt(S_max / (L * c / sig**2) + E0)
        
        ax[1].scatter(en, ent, label = 'XXZ - $S(E)$')
        # ax[1].scatter(Emax, e_fit(Emax, S_max, c), color = 'green')
        # ax[1].scatter(Emax, e_fit(Emax, c), color = 'green')

        # ax[1].plot(en, e_fit(en, S_max, c), color = 'red', label = f'${S_max:.3f} - L\cdot {c:.2f}(E-({E0:.3f}))^2 / ({sig:.3f})^2$')
        ax[1].plot(en, e_fit(en, c), color = 'red', label = f'${S_max:.3f} - L\cdot {c:.2f}(E-({E0:.3f}))^2 / (2{sig:.3f})^2$')

        # ---------------------------- legends ----------------------------
        ax[0].legend(fontsize = 16)
        ax[1].legend(fontsize = 16)
        
        E_take = Emax / 2
        energies_sig = np.arange(-E_take, E_take, 1e-4)
        Edown = en[av_idx - int(frac * N / 2)]
        Eup = en[av_idx + int(frac * N / 2)]

        df_save.iloc[i] = [sig_other, sig, E0, c, S_max, np.max(ent),
                           S_max + analytic_S(c, sig, E0, L, None, None), np.mean(ent),
                           np.mean(ent[av_idx - int(frac * N / 2) : av_idx + int(frac * N / 2)]),
                           S_max + analytic_S(c, sig, E0, L, Edown, Eup)]
        
        # ---------------------------- append values ----------------------------
        points[L] += list(en/sig)
        to_fig[L] += list(S_max - ent)
        
        if L == Ls[-1]:
            energy_concat[Nh:Nh+N] = en
            entro_concat[Nh:Nh+N] = ent
            Nh += N
        
        # ---------------------------- ADD PLOTS OF ANALYTIC INTEGRAL -------------------------------
        fractions = np.arange(0.01, 0.45, 5e-2)
        for k, f in enumerate(fractions):
            e = int(f * len(en))

            energies = en[av_idx-e//2:av_idx+e//2]
            if(av_idx - e//2 < 0):
                print("AAAAAAA")
                break
            state_num = dos_fit_L(L)(energies, E0, sig)
            conv = e_fit(energies, c) * state_num
            Edown = energies[0]
            Eup = energies[-1]
            
            real_mean = maximal_entropy(0.5, L) - np.mean(ent[av_idx-e//2:av_idx + e//2])
            integral = maximal_entropy(0.5, L) - np.trapz(conv, x=energies) / np.trapz(state_num, x=energies)
            int_a = maximal_entropy(0.5, L) - (S_max + analytic_S(c, sig, E0, L, Edown, Eup))
            if k == 0:
                ax[2].scatter(f, real_mean, color = 'red', label = f'XXZ, L=${L}$')
                ax[2].scatter(f, integral, color = 'green', label = f'Fit')
                ax[2].scatter(f, int_a, color = 'blue', label = 'Analytical', facecolor = 'None', s = 80)
            else:
                ax[2].scatter(f, real_mean, color = 'red')
                ax[2].scatter(f, integral, color = 'green')
                ax[2].scatter(f, int_a, color = 'blue', facecolor = 'None', s = 80)

        # ax[2].axhline(S_max, label = r'$S^{\rm max}$')
        #ax[2].set_yscale('log')
        ax[2].set_xlabel(r'$\nu$', fontsize = 16)
        ax[2].set_ylabel(r'$\langle S_A\rangle _n - \bar S_A^\nu$')
        ax[2].legend(fontsize = 16)        
        # ax[2].set_ylim(0, 0.4)
        plt.savefig(folder_save + model_shorts[i] + ".png")
        if i != end - 1:
            plt.close()
    # ---------------------------- save the dataframe ----------------------------
    df_save.to_csv(folder_save + "parameters.csv")
df_save

In [ ]:
power = True
####################### PLOT ##########################
fig, ax = plt.subplots(1, figsize = (10, 10), dpi = 300)
color_big = 'green'
ax.scatter(energy_concat, entro_concat, color = color_big, label = f"XXZ, L=${Ls[-1]}$")
ax.set_xlabel("$E/L$", fontsize = 16)
ax.set_ylabel("$S(E)$", fontsize = 16)
ax.set_ylim(1.5, page_result(2**(Ls[-1]//2), 2**(Ls[-1]//2)))
ax.set_xlim(-0.7, 0.7)
ax.legend(fontsize = 16)
ax.set_title(r"$S(E) = S^{\rm max} - c E^{2}$, $c = \tilde{c} / \sigma ^2$, $\sigma = \tilde{\sigma} \sqrt{L}$, $E = \tilde{E} L$", fontsize = 16)
if power:
    left, bottom, width, height = [0.3, 0.2, 0.4, 0.3]
else:
    left, bottom, width, height = [0.262, 0.2, 0.5, 0.4]
    
ax2 = fig.add_axes([left, bottom, width, height])
ax2.set_xlabel(r"$\tilde{E}_{\alpha}\sqrt{L}/\tilde{\sigma}_{\alpha}$" if not power else r"$\tilde{E}_{\alpha}^2L/\tilde{\sigma}_{\alpha}^2$")
ax2.set_ylabel(r"$S^{\rm max} - S_A$")
for L in Ls:
    np.save(save_dir + kPSep + "plot_data" + kPSep + f"energies_L={L},J=2.0,d=0.2_num", np.array(points[L]))
    np.save(save_dir + kPSep + "plot_data" + kPSep + f"points_L={L},J=2.0,d=0.2_num", to_fig[L])

    ax2.scatter((np.array(points[L])[::3] * np.sqrt(L))**2 if power else np.array(points[L])[::3] * np.sqrt(L), to_fig[L][::3],
                color = next(colors_ls_cyc) if L < Ls[-1] else color_big, rasterized=True,
                marker=next(markers), label = f"L=${L}$")
ax2.set_ylim(-0.01, None)
# plot insets
x = np.linspace(-1*np.sqrt(Ls[-1]), 1*np.sqrt(Ls[-1]))
y = x**2
a = 0.66
y2 = a * x**2
a2 = 0.33
y3 = a2 * x**2
# plot guide to an eye
ax2.plot(x**2 if power else x, y3, color = next(colors_ls_cyc), label = r"$\frac{1}{3}x^2$", lw = 2)
ax2.plot(x**2 if power else x, y2, color = 'black', label = r"$\frac{2}{3}x^2$", lw = 4)
ax2.plot(x**2 if power else x, y, color = next(colors_ls_cyc), label = "$x^2$", lw = 2)

# df_save.head     
ax2.legend(loc = 'lower left' if not power else 'lower right', fontsize = 14)

plt.savefig(save_dir + kPSep + "plot_data" + kPSep + f"analytic_entropies_{'lin' if not power else 'pow'}.png", bbox_inches = 'tight', pad_inches=0.02) 

# df_sweep

# PARAMETERS SWEEP FOR MOST CHAOTIC POINT

In [ ]:
N = 20
su=True

# directory = save_dir + f"{kPSep}xyz_su_bc=0{kPSep}resultsXYZ{N}{kPSep}"
param_a = 'da'
param_b = 'Jb'
fractions = [100]

df_sweep = get_log_file(directory, read_log=False,su=su)
df_sweep = df_sweep[df_sweep[param_b] <= 4.0][df_sweep[param_a] <= 4.0]
df_sweep

In [ ]:
df_sweep[df_sweep['Jb'] == <]

### Save the concatenated vectors of entropies

In [ ]:

jbs = [0.1, 0.2, 1.0, 2.0, 3.0, 3.4, 3.8, 3.5, 4.0]
das = [0.1, 0.2, 0.6, 0.8, 1.0, 2.0]
df_entro = parse_dataframe(df_sweep, {'da' : das, 'Jb' : jbs})
directory_save = save_dir + f'{kPSep}plot_data{kPSep}newest{kPSep}entro_dist{kPSep}'
createFolder([directory_save])
print(directory_save, directory)

fig, ax = plt.subplots(2, figsize=(10,10))
frac = 100
# get the entropies
for (j2, d1) in [(3.8, 0.2)]:
    df_tmp = parse_dataframe(df_sweep, {'da' : [d1], 'Jb' : [j2]})
    print(df_tmp)
    #set_gap_ratios_df_log(df_tmp, directory, use_mls=False)
    
    # takes the names of the models
    model_shorts = df_tmp.loc[:,'model_short'].to_list()
    hilbert_sizes = df_tmp.loc[:,'Nh'].to_list()
    s = df_tmp.loc[:, 'sec'].to_list()#df_tmp.loc[:, 'k']
    entropies = []
    for i, short in enumerate(model_shorts):
        filename = short + '.h5'
        if not 'spectrum' in short:
            filename = short + ',spectrum_num=' + str(int(hilbert_sizes[i])) + '.h5'
        print(filename)
        # read all entropies
        ent, av_idx, Nh, _, dictionary = read_entropies(directory, filename, frac, verbose=False)
        entropies.append(ent.iloc[-1].to_numpy())
        
        # include imaginary sectors twice
        if s[i] != 'real':
            entropies.append(ent.iloc[-1].to_numpy())
            
    entropies = np.array(entropies).flatten()
    print(entropies)
    print(j2, d1, len(entropies))
    hist, bins = np.histogram(entropies, 100, density=True, normed = True)
    ax[0].stairs(hist, np.exp(bins), label = f'$J2={j2},\Delta _1={d1}$')
    ax[1].stairs(hist, bins, label = f'$J2={j2},\Delta _1={d1}$')

    np.save(directory_save + f'entro_dist_XXZ_L=20,J2={j2},d1={d1}', entropies)
    np.savetxt(directory_save + f'entro_dist_XXZ_L=20,J2={j2},d1={d1}.dat', entropies)
    
#plt.xlim(4, None)
# plt.xscale('symlog')
plt.legend()
# df_entro

### ENTROPIES ALL 

In [ ]:
################################### SAVE THE ENTROPIES AND ENERGIES CONCATENATED ######################################

fig, ax = plt.subplots(2, figsize = (5, 8), sharex=True)

directory_save = directory + f'{kPSep}plot_data{kPSep}newest{kPSep}all_entro{kPSep}'
createFolder([directory_save])

def get_dir(L : int, sym : str):
    direct = save_dir + f"{kPSep}Data{kPSep}"
    if sym == "obc":
        direct += f"OBC{kPSep}resultsXYZ{L:.1f}{kPSep}"
    else:
        direct += f"resultsXYZ{L:.1f}{kPSep}"
    return direct

j2 = 2.0
d1 = 0.2
L_obc = 20
L_pbc = 24
for i, sym in enumerate(['obc', 'pbc']):
    L = L_obc if i == 0 else L_pbc
    directory = get_dir(L, sym)
    df = get_log_file(directory, read_log=False, su=su)
    df = parse_dataframe(df, {param_b : [j2], param_a : [d1]})
    df = df.reset_index()
    #print(df)
    # takes the names of the models
    model_shorts = df.loc[:,'model_short'].to_list()
    hilbert_sizes = df.loc[:,'Nh'].to_list()
    ks = df.loc[:,'k'].to_list()
    ps = df.loc[:,'p'].to_list()
    xs = df.loc[:,'x'].to_list()
    secs = df.loc[:,'sec'].to_list()
    #print(hilbert_sizes)
    # s = df.loc[:, 'sec'].to_list()
    entropies = np.zeros((np.sum(hilbert_sizes)))
    energies = np.zeros((np.sum(hilbert_sizes)))
    last_idx = 0
    N = 0
    for j, short in enumerate(model_shorts):
        k = ks[j]
        p = ps[j]
        x = xs[j]
        filename = short + '.h5'
        if not 'spectrum' in short:
            filename = short + ',spectrum_num=' + str(int(hilbert_sizes[j])) + '.h5'
            
        # read all entropies
        ent, av_idx, Nh, _, dictionary = read_entropies(directory, filename, 1.0, verbose=False)
        #print(Nh)
        e = ent.iloc[-1].to_numpy().flatten()
        entropies[last_idx : last_idx + len(e)] = e
        np.save(directory_save + f'entro_k={k},p={p},x={x}_XXZ_{sym}_L={L},J2={j2},d1={d1}', e)

        # energies addition
        en = read_h5_file(directory, filename, 'energy').to_numpy().flatten()
        
        energies[last_idx : last_idx + len(en)] = en
        np.save(directory_save + f'energy_,k={k},p={p},x={x}_XXZ_{sym}_L={L},J2={j2},d1={d1}', en)
        
        last_idx += len(e)
        N += len(e)
        if secs[j] == 'imag':
            N += len(e)
        
    print("N=", N, "(L,L//2)=", binom(L, L//2))
    entropies = np.array(np.array(entropies).flatten(), dtype=np.float32).flatten()
    energies = np.array(np.array(energies).flatten(), dtype=np.float32).flatten()
    #print(energies)
    
    ax[i].scatter(energies, entropies, label = f'$J2={j2},\Delta _1={d1}$---' + ('obc' if i == 0 else 'pbc'))
    np.save(directory_save + f'entro_all_XXZ_{sym}_L={L},J2={j2},d1={d1}', entropies)
    # np.savetxt(directory_save + f'entro_all_XXZ_L={L},J2={j2},d1={d1}.dat', entropies)
    np.save(directory_save + f'energy_all_XXZ_{sym}_L={L},J2={j2},d1={d1}', energies)
    # np.savetxt(directory_save + f'entro_all_XXZ_L={L},J2={j2},d1={d1}.dat', entropies)
    #ax[i].legend()
    ax[i].set_xlabel("$E$")
    ax[i].set_ylabel("$S_n$")

In [ ]:
#entropies[np.sum(hilbert_sizes):]

### Save the averages

In [ ]:
# jbs = [0.5, 1.0, 2.0]
jbs = [2.0]
# das = [0.2, 0.6, 1.5]
das = [0.2]
param_a = 'da'
param_b = 'Jb'
# Ls_pbc = [16, 18, 20, 22, 24, 26]
# Ls_obc = [12, 14, 16, 18, 20]
Ls_pbc = [16, 20, 24]
Ls_obc = []
fractions = [100, 0.25, 0.5, 1.0]
# idx = lambda L: -1
idx_e = lambda L: int(L//4 - 1)

# directory_save = save_dir + f'{kPSep}plot_data{kPSep}newest{kPSep}size_scale_all_params{kPSep}'
directory_save = current_dir + f'{kPSep}plot_data{kPSep}other_f{kPSep}size_scale_all_params{kPSep}'
print(directory_save)
dir_all_sectors = directory_save + "all_sectors" + kPSep
createFolder([directory_save, dir_all_sectors])

################################### SYM ######################################
columns = ['S_tot', 'S_pred', 'S_re', 'S_im', 'S_max', 'S_min', 'S_max_re', 'S_max_im', 'S_min_re', 'S_min_im']
syms = ["obc", "pbc"]
for j2 in jbs:
    for d1 in das:
        # iterate params
        for frac in fractions:
            df_obc = pd.DataFrame(np.zeros((len(Ls_obc), len(columns))), columns = columns, index=Ls_obc)
            df_pbc = pd.DataFrame(np.zeros((len(Ls_pbc), len(columns))), columns = columns, index=Ls_pbc)
            #print(df_obc.shape)
            for l, Ls in enumerate([Ls_obc, Ls_pbc]):
                sym = syms[l]
                for L in Ls:
                    d = save_dir + f"{kPSep}Data{kPSep}"
                    if sym == "obc":
                        d += f"OBC{kPSep}resultsXYZ{L:.1f}{kPSep}"
                    else:
                        d += f"resultsXYZ{L:.1f}{kPSep}"
                    d = directory = current_dir + f"{kPSep}Data{kPSep}Data_Quater{kPSep}resultsXYZ{L:.1f}{kPSep}"

                    # read df
                    df_sweep = get_log_file(d, read_log=False, su=su)
                    df_sweep = parse_dataframe(df_sweep, {param_b : [j2], param_a : [d1]})#df_sweep[df_sweep[param_b] == j2][df_sweep[param_a] == d1]
                    df_sweep = df_sweep.reset_index()
                    entro_idx = idx_e(L) 
                    print(entro_idx)
                    # calc entropies
                    set_entropies_df_log(df_sweep, d, fractions=[frac], verbose=False, set_max=True, set_min=True, idx_e = entro_idx)
                    
                    # save csv
                    filename = dir_all_sectors + f"su_df_{sym}_nu={frac}_=L={L},J2={j2:.1f},d1={d1:.1f}.csv"
                    df_sweep.to_csv(filename)
                    
                    S = 0
                    S_r = 0
                    S_i = 0
                    S_max_r = df_sweep[df_sweep['sec']=='real']['S_max'].max()
                    S_min_r = df_sweep[df_sweep['sec']=='real']['S_min'].min()
                    S_max_i = df_sweep[df_sweep['sec']=='imag']['S_max'].max() if len(df_sweep[df_sweep['sec']=='imag']['S_max']) != 0 else 0
                    S_min_i = df_sweep[df_sweep['sec']=='imag']['S_min'].min() if len(df_sweep[df_sweep['sec']=='imag']['S_min']) != 0 else 1e13
                    S_pred = maximal_entropy(0.25, L)
                    # get the averages
                    Nh = 0
                    for idx, row in df_sweep.iterrows():
                        n = int(row["spectrum_num"])
                        sector = row['sec']
                        #print(sector)
                        S_tmp = float(row[f'S_f={frac:.3f}'])
                        if sector == 'real':
                            S_r += n * S_tmp
                            S += n * S_tmp
                            Nh += row['Nh']
                        else:
                            S_i += n * S_tmp * 2.
                            S += n * S_tmp * 2.
                            Nh += row['Nh'] * 2
                    S_r /= df_sweep[df_sweep['sec'] == 'real']['Nh'].sum()
                    if l == 1:
                        n = df_sweep[df_sweep['sec'] == 'imag']['Nh'].sum() * 2.0
                        S_i /= n
                    S /= Nh
                    
                    if l == 0:
                        df_obc.loc[L] = [S, S_pred, S_r, S_i, max(S_max_r, S_max_i), min(S_min_r, S_min_i), S_max_r, S_max_i, S_min_r, S_min_i]
                    else:
                        df_pbc.loc[L] = [S, S_pred, S_r, S_i, max(S_max_r, S_max_i), min(S_min_r, S_min_i), S_max_r, S_max_i, S_min_r, S_min_i]
                        
                # dir to save
                dir_nu = directory_save#current_dir + kPSep + "plot_data" + kPSep + "new" + kPSep + "size_scale_all_params" + kPSep + f"nu={frac:.2f}" + kPSep
                createFolder([dir_nu])
                filename = dir_nu + f"su_sizes_{sym}_nu={frac},J2={j2:.1f},d1={d1:.1f}"
                
                if l == 0:
                    df_obc.to_csv(filename + ".csv")
                    np.save(filename, df_obc.to_numpy())
                else:
                    df_pbc.to_csv(filename + ".csv")
                    np.save(filename, df_pbc.to_numpy())
                
df_sweep

## GAP RATIO SWEEP

In [ ]:
set_gap_ratios_df_log(df_sweep, directory, use_mls=False)

In [ ]:
df_sweep

In [ ]:
directory_ratios = save_dir + f'{kPSep}plot_data{kPSep}newest{kPSep}ratios{kPSep}'
createFolder([directory_ratios])
group_columns = ['gapratios']
log_grouped = log_group_two_params(df_sweep, col_a=param_a, col_b=param_b, columns = group_columns)
log_grouped[['Jb','gapratios']].to_csv(directory_ratios + "su_gap_ratio_L=20.csv")
log_grouped

In [ ]:
log_grouped = log_grouped[log_grouped.index.isin([0.2, 0.8, 1.4, 2.0, 3.0, 3.5, 3.8, 4.0])]

In [ ]:
#fig, axis = plt.subplots( nrows=1, ncols=2, figsize=(10,5), dpi = 100)
fig = plt.figure(figsize=(10,5), dpi = 200)
gs = GridSpec(nrows=1, ncols=2, width_ratios=[1, 1])
axis = []
axis.append(fig.add_subplot(gs[0]))
axis.append(fig.add_subplot(gs[1]))

## --- LEFT PANEL
# Jb, da
points_to_plot = [(3.7, 0.2), (3.5, 0.2), (3.8, 0.2), (2.0, 0.2), (4.0, 0.2), (3.5, 0.8)]
xpoi = np.linspace(0, 1, 100)



#axis[0].plot(xpoi, goe(xpoi), linestyle='--', color='black')
axis[0].plot(xpoi, 27 / 4 * (( xpoi + xpoi**2 ) / ( 1 + xpoi + xpoi**2 )**(5/2)), linestyle='--', color='black')
axis[0].plot(xpoi, 2 / (1 + xpoi)**2, linestyle='--', color='red')

for [J2, da] in points_to_plot:
    tmp = df_sweep[df_sweep['da']==da]
    tmp = tmp[tmp['Jb'] == J2]

    level_stats = read_gap_ratios_df_log(tmp, directory)
    np.save(directory_ratios + f"su_gap_ratio_hist_L=20,J2={J2:.1f},d1={da:.1f}", np.array(level_stats))
    
    hist, bins = np.histogram(level_stats, bins=40, density=True)
    axis[0].stairs(hist, bins, label=r"$J_2=%.1f,\Delta _1=%.1f$"%(J2, da))
axis[0].set_xlim(0,1)    
axis[0].set_xlabel("$r$", size = 16)
axis[0].set_ylabel("$P(r)$", size = 16)
axis[0].legend(loc = 'upper right', frameon=False, fontsize=16, handletextpad=0.25, handlelength = 1.25, bbox_to_anchor=(1.02, 1.02))


## --- RIGHT PANEL
symbols = ['^', '+', 'o', 's']

label_b = '$\Delta _1$'
label_a = '$J_2$'

# plot
for i,a in enumerate(log_grouped.index.unique()):
    # use col_b as x values and col_a as labels
    axis[1].plot(log_grouped.loc[a, param_b], log_grouped.loc[a,'gapratios'], label = f'{label_b}$={a}$', linewidth=0, marker = symbols[i], markersize=8)
    
axis[1].set_ylim(0.38, 0.54)
axis[1].axhline(y=0.5307, ls='--', color='black', label = '$r_{GOE}$', linewidth=3)
axis[1].axhline(y=0.3867, ls='--', color='black', label = '$r_{POI}$', linewidth=3)

axis[1].set_xlabel(label_a, size = 16)
axis[1].set_ylabel('$r$', size = 16)
axis[1].set_xlim(0.1,4.0)
#axis[1].set_ylim(0.47, 0.54)
axis[1].legend(fontsize = 16)

## --- COMSETICS

for ax in list(axis):
    ax.set_xscale('linear')    
    ax.set_yscale('linear')
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.tick_params(axis="both",which='major',direction="in",length=6)
    ax.tick_params(axis="both",which='minor',direction="in",length=3)

axis[0].annotate(r"$(a)$", xy=(0.85, 0.08), fontsize=18, xycoords='axes fraction')
axis[1].annotate(r"$(b)$", xy=(0.85, 0.08), fontsize=18, xycoords='axes fraction')
fig.subplots_adjust(wspace=0.3, hspace=0.3)

## INFORMATION ENTROPY SWEEP

In [ ]:
set_info_entro_df_log(df_sweep, directory)

In [ ]:
group_columns = ['S_info']
log_grouped = log_group_two_params(df_sweep, col_a=param_a, col_b=param_b, columns = group_columns, rescale = False)
log_grouped

In [ ]:
log_grouped = log_grouped[log_grouped.index.isin([0.3, 0.5, 0.6, 0.8, 0.9, 1.1])]

In [ ]:

label_b = '$\Delta _1$'
label_a = '$J_2$'

# plot
fig, ax = plt.subplots(1, figsize = (5, 5), dpi=200)
fig.tight_layout()
for a in log_grouped.index.unique():
    # use col_b as x values and col_a as labels
    ax.plot(log_grouped.loc[a, param_b], log_grouped.loc[a,'S_info'], label = f'{label_b}$={a}$', linewidth=0, marker = next(markers), markersize=8)
    
#ax.axhline(goe, label = '$r_{GOE}$', linestyle = "--", linewidth=3, color='black')
ax.set_xlabel(label_a, size = 16)
ax.set_ylabel('$S_{info}/\log ( 0.48*\mathcal{D})$', size = 16)
ax.set_xlim(0.1,1.1)
#ax.set_ylim(1.015, 1.030)
#ax.set_yscale('log')
ax.tick_params(axis='both', which='both', rotation = 0, labelsize=16)
ax.tick_params(axis="both",which='major',direction="in",length=6)
ax.tick_params(axis="both",which='minor',direction="in",length=3)
#plt.ticklabel_format(axis='y', style='sci', scilimits=(3,3))
#ax.set_title(title)
ax.legend(fontsize = 16)

## ENTANGLEMENT ENTROPY SWEEP

In [ ]:
set_entropies_df_log(df_sweep, directory, fractions=fractions, verbose=False, set_max=True, set_min=True)

In [ ]:
group_columns = [f'S_f={i:.3f}' for i in fractions]
log_grouped = log_group_two_params(df_sweep, col_a=param_a, col_b=param_b, columns = group_columns)
log_grouped

In [ ]:
log_grouped = log_grouped[log_grouped.index.isin([0.2, 0.8, 1.4, 2.0])]

In [ ]:

label_b = '$\Delta _1$'
label_a = '$J_2$'

directory_e = directory + f'{kPSep}plot_data{kPSep}newest{kPSep}entro{kPSep}'
createFolder([directory_e])

points_to_plot = [(2.0, 0.2), (0.2, 0.2), (1.0, 1.0), (0.1, 2.0), (3.0, 0.2)]
# for j2, da in points_to_plot:
#     tmp = df_sweep[df_sweep['da']==da]
#     tmp = tmp[tmp['Jb'] == j2]
#     print(tmp[group_columns[0]])
#     np.save(directory_ratios + f"su_entro_hist_L=20,J2={j2:.1f},d1={da:.1f}", np.array(level_stats))


# plot
fig, ax = plt.subplots(1, figsize = (5, 5), dpi=200)
fig.tight_layout()
for a in log_grouped.index.unique():
    # use col_b as x values and col_a as labels
    ax.plot(log_grouped.loc[a, param_b], log_grouped.loc[a,group_columns[0]] / N, label = f'{label_b}$={a}$', linewidth=0, marker = next(markers), markersize=8)
    
#ax.axhline(goe, label = '$r_{GOE}$', linestyle = "--", linewidth=3, color='black')
ax.set_xlabel(label_a, size = 16)
ax.set_ylabel('$S / L$', size = 16)
ax.set_xlim(0.0,4.0)
ax.set_ylim(0.3, None)
#ax.set_yscale('log')
ax.tick_params(axis='both', which='both', rotation = 0, labelsize=16)

ax.tick_params(axis="both",which='major',direction="in", length=6)
ax.tick_params(axis="both",which='minor',direction="in", length=3)

#plt.ticklabel_format(axis='y', style='sci', scilimits=(3,3))
#[i for i in range(10)]
a = (i for i in range(10))
#ax.set_title(title)
ax.legend(fontsize = 16)

In [ ]:

label_b = '$\Delta _a$'
label_a = '$J_b / J_a$'

# plot
fig, ax = plt.subplots(1, figsize = (10, 8))

for a in log_grouped.index.unique():
    # use col_b as x values and col_a as labels
    ax.plot(log_grouped.loc[a, param_b], page_val(N) - log_grouped.loc[a,group_columns[0]], label = f'{label_b}$={a}$', linewidth=0, marker = next(markers), markersize=8)
    
#ax.axhline(goe, label = '$r_{GOE}$', linestyle = "--", linewidth=3, color='black')
ax.set_xlabel(label_a, size = 18)
ax.set_ylabel('$S_{page} - S$', size = 18)
ax.set_xlim(0.05,1.15)
#ax.set_ylim(1.015, 1.030)
#ax.set_yscale('log')
ax.tick_params(axis='both', which='both', rotation = 0, labelsize=14)
#plt.ticklabel_format(axis='y', style='sci', scilimits=(3,3))
#ax.set_title(title)
ax.legend(fontsize = 18)

## EIGENSTATES COEFFICIENTS DISTRIBUTION

#### GET STATES

In [ ]:
params_a = df_sweep[param_a].unique().tolist()
params_b = df_sweep[param_b].unique().tolist()


'''
Real - 0
Imag - 1
'''
def get_states(a, b):
    states = []
    df_tmp = df_sweep[df_sweep[param_a] == a][df_sweep[param_b]==b]
    for sec in ['real', 'imag']:
        df_tmp_sec = df_tmp[df_tmp['sec'] == sec]
        states.append(read_eigenstates_df_log(df_tmp_sec, directory, sec))
    return states    

np.array(get_states(0.2, 2.0)[0]).shape

In [ ]:
df_sweep2=df_sweep#[df_sweep['S_max'] != 0]
df_sweep2

In [ ]:
# calculate gaussianities
# df_sweep['gapratios'] = np.zeros(len(df_sweep))
group_columns = ['gapratios'] + [f'S_f={i:.3f}' for i in fractions]  
log_grouped = log_group_two_params(df_sweep, col_a=param_a, col_b=param_b, columns = group_columns)

tmp = df_sweep2.groupby(['da','Jb']).max().sort_values(['da','Jb']).reset_index()
log_grouped['entro_min'] = np.zeros(len(log_grouped))
log_grouped['entro_max'] = np.zeros(len(log_grouped))
log_grouped = log_grouped.reset_index().sort_values(['da', 'Jb']).reset_index()
# set the values again
log_grouped['entro_min'] = tmp['S_min']
log_grouped['entro_max'] = tmp['S_max']
log_grouped.drop(['index','Nh', 'Nh2'], axis = 1, inplace=True)
log_grouped['gaussianity'] = np.zeros(len(log_grouped))
log_grouped.to_csv(directory +  f'{kPSep}plot_data{kPSep}newest{kPSep}param_sweep.csv')

for i, row in log_grouped.iterrows():
    try:
        da = row['da']
        jb = row['Jb']
        print(da, jb)
        states = get_states(da, jb)
        states_r = np.array(states[0])
        states_i = np.array(states[1])
        states_i = np.concatenate([states_i.real, states_i.imag])
        s = np.concatenate([states_r, states_i])
        g = gaussianity(np.abs(s))
        print("\t",g)
        log_grouped.loc[i]['gaussianity'] = g
    except:
        pass
log_grouped.to_csv(directory +  f'{kPSep}plot_data{kPSep}newest{kPSep}param_sweepg.csv')

# log_grouped['entropy_min'] = df_sweep.groupby(['da','Jb']).max().reset_index('Jb')['S_min']
log_grouped


#### PLOT

In [ ]:

#fig_dist, ax_dist = plt.subplots( nrows=1, ncols=2, figsize=(12,5), dpi = 200)
fig_dist = plt.figure(figsize=(10,5), dpi = 200)
fig_dist.tight_layout(pad=2.0)

gs = GridSpec(nrows=1, ncols=2, width_ratios=[1, 1])
ax_dist = []
ax_dist.append(fig_dist.add_subplot(gs[0]))
ax_dist.append(fig_dist.add_subplot(gs[1]))

width = 0.4
height = 0.4
ax_inset = []
# insert insets
for ax in ax_dist:    
    ax_sub = add_subplot_axes(ax, [0.98 - width, 0.98 - height, width, height])
    ax_inset.append(ax_sub)


print("hx\tk\tp\tshape\t\tvar diff")
def plotter(which, params_a, params_a_dist, b, which_dist = 'chi', label_a = "\\Delta ^a"):
    ax_all = [ax_inset[which], ax_dist[which]]

    gauss = []
    coeffs = np.array(0)
    for a in params_a:
        colors_inset = {
            14: 'blue', 
            16: 'orange', 
            18: 'green', 
            20: 'red'
            }
        
        # function to rescale vector
        rescale_vec = lambda x : (x - np.mean(x))/np.sqrt(np.var(x))

        # get the states
        vec = get_states(a, b)[which]
        
        if which == 0:
            coeffs = np.abs(vec)
        else:
            states_r = vec.real
            states_i = vec.imag
            
            print(f'\n\t->da={a}\n\t\t->var_r={np.var(states_r)},\t->var_i={np.var(states_i)}')
            print(f'\t\t->mu_r={np.mean(states_r)},\t->mu_i={np.mean(states_i)}')
            print(f'\t\t->gauss_r={gaussianity(np.abs(states_r)) - np.pi / 2 :.3e},\t->gauss_i={gaussianity(np.abs(states_i)) - np.pi / 2 :.3e}')
            # different types
            if which_dist == 'chi':
                coeffs = np.sqrt(np.square(states_i) + np.square(states_r))
            elif which_dist == 'chi2':
                coeffs = np.square(states_i) + np.square(states_r)
            elif which_dist == 'folded':
                coeffs = np.abs(states_i) + np.abs(states_r)
        
        hist, edge = np.histogram(coeffs, bins=200, density=True)
        #ax_dist[which].plot(edge[:-1], hist, marker='o', markersize=1, linewidth=0.5, label=r"$h^z=%.1f$"%(hx))
        if a in params_a_dist:
            ax_dist[which].stairs(hist, edge, label=r"$%s=%.1f$"%(label_a, a))
        
        # get the gaussianity
        gauss = gaussianity(coeffs)
        print("\tGauss=", gauss)
        ax_inset[which].scatter(a, gauss, color='black', s=10)



    #fun_lab = r"$f(c_n)=|c_n/\sigma|$" if which == 0 else r"$f(c_n)=\sqrt{\left(\Re{c_n}/\sigma_{\Re{c_n}}\right)^2 + \left(\Im{c_n}/\sigma_{\Im{c_n}}\right)^2}$"
    #ax_dist[which].set_title(fun_lab)
    ylab = "P(z)" if which == 0 else " "
    xlab = "$z=|c_n|\\cdot\\sqrt{D}$" if which == 0 else "$z=|c_n|\\cdot\\sqrt{2D}$"
    ax_dist[which].set_xlabel(xlab, size = 16)
    ax_dist[which].set_ylabel(ylab, size = 16)
    ax_dist[which].set_xlim((None,None))
    ax_dist[which].set_ylim((None,None))
    
    if which == 0:
        ax_dist[which].legend(loc = 'lower left', frameon=False, fontsize=16, handletextpad=0.25, handlelength = 1.25, bbox_to_anchor=(-0.02, -0.02))

    # inset 
    ax_inset[which].set_xlabel(f"${label_a}$", size = 12)
    ax_inset[which].set_ylabel("$\\Gamma_\\Psi$", size = 12)
    ax_inset[which].set_xlim((None,None))
    ax_inset[which].set_ylim((None,None))

    # set ticks etc
    for ax in list(ax_all):
        ax.set_xlim(0.0, 0.15)
        ax.set_xscale('linear')    
        ax.set_yscale('linear')
        ax.tick_params(axis="both",which='major',direction="in",length=6)
        ax.tick_params(axis="both",which='minor',direction="in",length=3)
        ax.set_xlim(0.0, 2.1)
        
    ax_all[0].set_xlim(0., 2.1)
    # val of gaussianity
    val = np.pi / 2 if which == 0 else 4. / np.pi
    valname = r"$\pi/2$" if which == 0 else r"$4/\pi$"
    
    ax_all[0].axhline(y=val, ls='--', color='black')
    yticks = [val, 1.8, 2.0, 2.2] if which == 0 else [val, 1.5, 1.8]   
    tick_labels = ["%.1f"%ytic if ytic != val else valname for ytic in yticks]
    ax_all[0].set_yticks(yticks)
    ax_all[0].set_yticklabels(tick_labels)
    if which == 0:  ax_all[1].set_ylim(1.3, 2.4)
    else:           ax_all[1].set_ylim(1.1, 2.0)
    ax_all[0].xaxis.set_minor_locator(plt.NullLocator())
    ax_all[0].yaxis.set_minor_locator(plt.NullLocator())

    # distributioon
    ax_all[1].set_xscale('linear')
    ax_all[1].set_yscale('log')
    aa = np.linspace(-100.0, 100, 20000)
    mu=0.0
    sigma=1.
    ax_all[1].set_ylim(1e-6, 3.0)
    ax_all[1].set_xlim(0, 10)
    if which == 0:    
        ax_all[1].plot(aa, 2. / np.sqrt(2*np.pi) * np.exp( - (aa)**2 / (2)), linestyle='--', color='black')
        ax_all[1].plot(aa, np.exp(-1.2*aa), linestyle='--', color='red')
    else: 
        fit_val = np.array(0)
        if which_dist == 'chi':
            fit_val = sum_of_squares_normals_sqrt(aa)
        elif which_dist == "chi2":
            fit_val = sum_of_squares_normals(aa)
        elif which_dist == 'folded':
            fit_val = sum_of_folded_normals(aa, 1.0, 1.0)
        # plot the fit
        ax_all[1].plot(aa, fit_val, linestyle='--', color='black')
        

#plotter(0, params_a=params_a, params_a_dist = [0.1, 0.5, 1.1], b = 2.0)
plotter(1, params_a=params_a, params_a_dist = [0.1, 0.5, 1.1], b = 2.0, which_dist = 'chi')


# Arbitrary number of particles

The average entanglement entropy of a uniformly distributed pure state in $\mathcal{H}$ restricted to subsystem A is given by the Page formula:
$$
    \langle S_A \rangle = 
    \left\{
    \begin{array}{lc}
        \Psi(d_Ad_B+1) - \Psi(d_B + 1) - \frac{d_A - 1}{2d_B}, & d_A\leq d_B \\
        \Psi(d_Ad_B+1) - \Psi(d_A + 1) - \frac{d_B - 1}{2d_A}, & \text{otherwise}
    \end{array}
    \right.,
$$

where $\Psi(x) = \Gamma ' (x) / \Gamma (x)$ is a digamma function.

Going to a thermodynamic limit, in big systems we can fix a fraction $f = V_A/V$ so what we get is:
$$
    \langle S_A \rangle = f V\ln 2 - 2 ^{-|1-2f|V-1} + O(2^{-V}),
$$
where we see the volueme law. For $f \neq \frac{1}{2}$, the second term is an exponentially small correction and becomes $-\delta _{f,1/2}$ as $V\rightarrow \infty$.

The variance is given for $d_A \leq d_B$:
$$
(\Delta S_A )^2 = \frac{d_A + d_B}{d_Ad_B + 1}\Psi ' (d_B + 1) - \Psi ' (d_Ad_B + 1) - \frac{(d_A - 1)(d_A + 2d_B - 1)}{4d_B^2(d_Ad_B + 1)},
$$
which again in the termodynamic limit becomes:
$$
(\Delta S_A )^2 = \left(\frac{1}{2}-\frac{1}{4}\delta_{f,1/2}\right)2^{-(1+|1-2f|)V}+o\left(2^{-(1+|1-2f|)V}\right).

In [ ]:
directory=current_dir + f"{kPSep}Data{kPSep}Data{kPSep}"
bins = 50
params = {'hz':[0.2], 'hx':[0.3], 'da' : [0.3], 'k' : [1], 'p' : [1]}
plot_eig_hist([16], 100, directory, params, bins = bins, dens = True)

In [ ]:
L=[14, 16, 18, 20]
fractions=[100, 0.2, 0.1]
read_log=False
su2=False
verbose=False
params = {'hz':[0.2], 'hx':[0.3], 'da' : [0.3], 'Jb' : [2.0]}
directory=current_dir + f"{kPSep}Data{kPSep}Data{kPSep}"
all_df, all_cols = plot_difference_log(L=L, fractions=fractions,
                                       directory=directory,
                                       params = params,
                                       ylim=[3e-2, 0.2],
                                       read_log=False, su2=su2, verbose=verbose)
all_df

# ENTROPIES FULL

In [ ]:
Ls=[14, 16, 18, 20]
bc = 0
k = 0
p = 1.
x = 1.
frac_c = 0.2
read_log=False
su2=False

dfs = []
ens = []
directory_l = lambda l: directory + f"{kPSep}resultsXYZ{l}{kPSep}"
for L in Ls:
    direc = directory_l(L)
    log = get_log_file(direc, read_log=read_log, su2=su2)
    log = parse_dataframe(log, {'k':[k], 'p':[p], 'x':[x], 'Jb':[2.0], 'hz' : [0.2]})
    name = log[(log['k'] == k) & (log['p']==p) & (log['x']==x)]['model_short'].iloc[0]
    print(name)
    file = name + ".h5"
    entropies, av_idx, _, _, _ = read_entropies(direc, file, 1.0, verbose=True)
    energies = read_h5_file(direc, file, 'energy').to_numpy()
    dfs.append(entropies)
    ens.append(energies)

plot_df_together(Ls, dfs, ens, directory, su2)
plot_rescale_df(Ls, dfs, ens, directory, su2)
maxima_df(Ls, dfs, ens, directory, su2=su2, bin_num=100)